# Importing Libraries

In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import os

# Image Processing

In [5]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 5 

# Set up data augmentation for training images
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2
)

# Only rescale validation images (no tampering!)
val_datagen = ImageDataGenerator(rescale=1./255)

# Connect generators to your physical folders
train_generator = train_datagen.flow_from_directory(
    'tomato/train',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    'tomato/valid',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 25851 images belonging to 11 classes.
Found 6683 images belonging to 11 classes.


# Loading The Model

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze baseline layers

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)

num_classes = train_generator.num_classes 
predictions = Dense(num_classes, activation='softmax')(x)

my_custom_model = Model(inputs=base_model.input, outputs=predictions)
my_custom_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

my_custom_model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,919,499 (11.14 MB)

 Trainable params: 661,515 (2.52 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
history = my_custom_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

my_custom_model.save('tomato_model.h5')
print("Model compiled and saved successfully as tomato_model.h5!")

Epoch 1/5
808/808 ━━━━━━━━━━━━━━━━━━━━ 651s 803ms/step - accuracy: 0.7426 - loss: 0.7464 - val_accuracy: 0.7947 - val_loss: 0.5921
Epoch 2/5
808/808 ━━━━━━━━━━━━━━━━━━━━ 591s 731ms/step - accuracy: 0.8376 - loss: 0.4641 - val_accuracy: 0.8266 - val_loss: 0.5046
Epoch 3/5
808/808 ━━━━━━━━━━━━━━━━━━━━ 543s 671ms/step - accuracy: 0.8657 - loss: 0.3814 - val_accuracy: 0.8558 - val_loss: 0.4191
Epoch 4/5
808/808 ━━━━━━━━━━━━━━━━━━━━ 629s 779ms/step - accuracy: 0.8866 - loss: 0.3237 - val_accuracy: 0.8625 - val_loss: 0.3983
Epoch 5/5
808/808 ━━━━━━━━━━━━━━━━━━━━ 582s 720ms/step - accuracy: 0.8983 - loss: 0.2871 - val_accuracy: 0.8682 - val_loss: 0.4049


Model compiled and saved successfully as tomato_model.h5!


In [ ]:
classes = sorted(list(train_generator.class_indices.keys()))
print(classes)